# 03 · 모델 테스트 및 추론 검증

이 노트북에서는 학습된 모델을 불러와 샘플 이미지를 평가하고, Top-3 예측 결과를 확인합니다.



In [ ]:
# !pip install -q torch torchvision pillow numpy matplotlib

from pathlib import Path
import json
import torch
from torchvision import transforms
from PIL import Image

MODEL_PATH = Path('../models/best_model.pth')
CLASS_MAP_PATH = Path('../models/class_mapping.json')

checkpoint = torch.load(MODEL_PATH, map_location='cpu')
class_names = checkpoint['class_names']



In [ ]:
from torchvision import models

model = models.efficientnet_b0(weights=None)
model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, len(class_names))
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

preprocess = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [ ]:
import torch.nn.functional as F

def predict(image_path: Path, topk: int = 3):
    image = Image.open(image_path).convert('RGB')
    tensor = preprocess(image).unsqueeze(0)
    with torch.no_grad():
        outputs = model(tensor)
        probs = F.softmax(outputs, dim=1)
    top_probs, top_indices = probs.topk(topk)
    results = []
    for prob, idx in zip(top_probs.squeeze(), top_indices.squeeze()):
        results.append({
            'label': class_names[idx],
            'confidence': float(prob.item())
        })
    return results

sample_image = Path('../data/seoul_museums/국립현대미술관 서울관/001.jpg')
predict(sample_image)



In [ ]:
import matplotlib.pyplot as plt

plt.imshow(Image.open(sample_image))
plt.axis('off')
for rank, entry in enumerate(predict(sample_image), start=1):
    print(f"Top{rank}: {entry['label']} ({entry['confidence']*100:.2f}%)")

